# 20 - Historical LSMC / Option B1

Historical perfect-foresight LP bridge on actual 2024-2026 GB DA and system-price paths. This is a clairvoyant upper benchmark, not a realized trading strategy. All inputs and assumptions are visible in this notebook.

In [ ]:
from __future__ import annotations

import copy
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as cfg
from src.config import ASSET, configure_asset_duration
from src.optimisation.perfect_foresight import solve_perfect_foresight

## Inputs and Assumptions

- Source files: corrected raw Elexon DA and SP parquet files.
- Date range: 2024-04-01 to 2026-04-25.
- WD60 path: DA plus clipped `(SP - DA)` basis at +/- GBP60/MWh.
- Durations: 1h and 2h for comparison with Modo public index.
- LP uses terminal SoC equal to initial SoC and deducts asset VOM.

In [ ]:
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

START = pd.Timestamp("2024-04-01")
END = pd.Timestamp("2026-04-25")
DT_H = 0.5
WD_CAP = 60.0
DURATIONS_H = [1.0, 2.0]
AS_OF = "2026-05-17"

cfg.VALID_ASSET_DURATIONS_H = (1.0, 2.0, 3.0, 4.0)

INPUTS = {
    "da_prices": RAW / "elexon_da_prices.parquet",
    "sp_prices": RAW / "elexon_sp_prices.parquet",
}
INPUTS

## Load and Align Historical Prices

DA rows are volume-weighted by settlement date/period if multiple provider rows are present. SP rows are averaged by settlement date/period. The final frame is an inner join, so every row has both DA and SP.

In [ ]:
def load_da() -> pd.DataFrame:
    df = pd.read_parquet(INPUTS["da_prices"])
    df["settlement_date"] = pd.to_datetime(df["settlement_date"])
    df = df[df["settlement_date"].between(START, END)].copy()
    if "volume_mwh" in df.columns:
        df["weighted_price"] = df["price_gbp_mwh"] * df["volume_mwh"].clip(lower=0.0)
        grouped = df.groupby(["settlement_date", "settlement_period"], as_index=False).agg(
            volume_mwh=("volume_mwh", "sum"),
            weighted_price=("weighted_price", "sum"),
            price_mean=("price_gbp_mwh", "mean"),
        )
        grouped["price_gbp_mwh"] = np.where(
            grouped["volume_mwh"] > 0,
            grouped["weighted_price"] / grouped["volume_mwh"],
            grouped["price_mean"],
        )
        return grouped[["settlement_date", "settlement_period", "price_gbp_mwh"]]
    return df[["settlement_date", "settlement_period", "price_gbp_mwh"]]


def load_sp() -> pd.DataFrame:
    df = pd.read_parquet(INPUTS["sp_prices"])
    df["settlement_date"] = pd.to_datetime(df["settlement_date"])
    df = df[df["settlement_date"].between(START, END)].copy()
    df = df.groupby(["settlement_date", "settlement_period"], as_index=False)["system_price"].mean()
    return df.rename(columns={"system_price": "sp_gbp_mwh"})


def aligned_prices() -> pd.DataFrame:
    da = load_da()
    sp = load_sp()
    frame = da.merge(sp, on=["settlement_date", "settlement_period"], how="inner")
    frame = frame.sort_values(["settlement_date", "settlement_period"]).reset_index(drop=True)
    frame["delta_sp_da"] = frame["sp_gbp_mwh"] - frame["price_gbp_mwh"]
    frame["wd60_gbp_mwh"] = frame["price_gbp_mwh"] + frame["delta_sp_da"].clip(-WD_CAP, WD_CAP)
    return frame


prices = aligned_prices()
coverage = {
    "rows": len(prices),
    "start": str(prices["settlement_date"].min().date()),
    "end": str(prices["settlement_date"].max().date()),
    "da_mean": float(prices["price_gbp_mwh"].mean()),
    "sp_mean": float(prices["sp_gbp_mwh"].mean()),
    "wd60_mean": float(prices["wd60_gbp_mwh"].mean()),
}
coverage

## Run Perfect-Foresight LP

Each stream is solved independently for each duration. Values are annualised over the exact number of half-hours in the aligned historical window.

In [ ]:
def run_case(price: np.ndarray, duration_h: float, stream: str, asset: dict) -> dict:
    result = solve_perfect_foresight(
        price,
        asset,
        dt_h=DT_H,
        terminal_soc_mwh=asset["soc_init_mwh"],
        vom_gbp_mwh=asset["vom_gbp_mwh"],
    )
    horizon_years = len(price) * DT_H / 8760.0
    return {
        "duration_h": duration_h,
        "stream": stream,
        "rows": int(len(price)),
        "horizon_years": horizon_years,
        "value_gbp": result.objective_gbp,
        "gbp_per_mw_year_k": result.objective_gbp / asset["power_mw"] / horizon_years / 1000.0,
        "cycles_equiv": result.cycles_equiv,
        "terminal_soc_mwh": float(result.soc_mwh[-1]),
        "price_mean_gbp_mwh": float(np.mean(price)),
        "price_min_gbp_mwh": float(np.min(price)),
        "price_max_gbp_mwh": float(np.max(price)),
    }


price_map = {
    "DA": prices["price_gbp_mwh"].to_numpy(dtype=float),
    "SP_perfect_foresight": prices["sp_gbp_mwh"].to_numpy(dtype=float),
    "WD60_perfect_foresight": prices["wd60_gbp_mwh"].to_numpy(dtype=float),
}

rows: list[dict] = []
for duration_h in DURATIONS_H:
    asset = copy.deepcopy(ASSET)
    configure_asset_duration(asset, duration_h)
    for stream, arr in price_map.items():
        print(f"Solving {stream} {duration_h:g}h on {len(arr):,} HH...")
        rows.append(run_case(arr, duration_h, stream, asset))

df_rows = pd.DataFrame(rows)
pivot = df_rows.pivot(index="stream", columns="duration_h", values="gbp_per_mw_year_k")
pivot.round(1)

## Save Outputs

In [ ]:
summary = {
    "as_of": AS_OF,
    "method": "Option B1 historical perfect-foresight LP",
    "start_date": str(prices["settlement_date"].min().date()),
    "end_date": str(prices["settlement_date"].max().date()),
    "wd_cap_gbp_mwh": WD_CAP,
    "dt_h": DT_H,
    "rows": rows,
}

out_json = PROCESSED / "historical_lsmc_b1_summary.json"
out_csv = PROCESSED / "historical_lsmc_b1_summary.csv"
out_prices = PROCESSED / "historical_lsmc_b1_prices.parquet"
out_png = PROCESSED / "historical_lsmc_b1_summary.png"

out_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")
df_rows.to_csv(out_csv, index=False)
prices.to_parquet(out_prices, index=False)

ax = pivot.T.plot(kind="bar", figsize=(8, 4.5))
ax.set_title("Historical B1 perfect-foresight LP on actual prices")
ax.set_xlabel("Battery duration (hours)")
ax.set_ylabel("GBPk/MW/year")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="", fontsize=8)
plt.tight_layout()
plt.savefig(out_png, dpi=140, bbox_inches="tight")
plt.show()

{
    "summary_json": str(out_json),
    "summary_csv": str(out_csv),
    "price_audit_parquet": str(out_prices),
    "chart_png": str(out_png),
}

## Convert Units and Compare to nr13

`nr20` reports historical B1 results in `GBPk/MW/year`. The nr13 method-comparison chart reports `GBPm/year` for the full 100 MW asset, so the conversion is:

```text
GBPm/year = GBPk/MW/year * asset_power_mw / 1000
```

The strict like-for-like comparison is `nr20 DA` against `nr13 Perfect foresight (DA energy)`. The SP and WD60 historical perfect-foresight rows are kept as context because they use actual imbalance/within-day information and are not the same methodology as nr13's non-anticipative or rolling methods.

In [ ]:
POWER_MW = float(ASSET["power_mw"])

# Keep the converted unit visible in the main B1 table and persist it for audit.
df_rows["value_gbp_annualized_m"] = df_rows["gbp_per_mw_year_k"] * POWER_MW / 1000.0
df_rows.to_csv(out_csv, index=False)

converted = df_rows.pivot(
    index="stream",
    columns="duration_h",
    values="value_gbp_annualized_m",
)
converted.round(2)

In [ ]:
def load_nb13_reference(durations_h: list[float]) -> pd.DataFrame:
    rows = []
    for duration_h in durations_h:
        label = f"{duration_h:g}h"
        path = PROCESSED / f"phase4_method_comparison_{label}.csv"
        if not path.exists():
            print(f"Missing nr13 reference: {path}")
            continue
        frame = pd.read_csv(path)
        frame["duration_h"] = duration_h
        rows.append(frame)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def build_nb13_comparison(df_rows: pd.DataFrame) -> pd.DataFrame:
    b1 = df_rows.rename(columns={"gbp_per_mw_year_k": "b1_gbp_per_mw_year_k"}).copy()
    b1["b1_value_gbp_annualized_m"] = b1["b1_gbp_per_mw_year_k"] * POWER_MW / 1000.0

    nb13 = load_nb13_reference(sorted(b1["duration_h"].unique()))
    reference_methods = [
        "Perfect foresight (DA energy)",
        "DA rolling intrinsic",
        "WD rolling intrinsic",
        "Forward simulation (LSMC)",
    ]
    nb13 = nb13[nb13["method"].isin(reference_methods)].rename(
        columns={
            "method": "nb13_method",
            "value_gbp_annualized_m": "nb13_value_gbp_annualized_m",
            "gbp_per_mw_year_k": "nb13_gbp_per_mw_year_k",
        }
    )

    direct = b1[b1["stream"].eq("DA")].merge(
        nb13[nb13["nb13_method"].eq("Perfect foresight (DA energy)")],
        on="duration_h",
        how="inner",
    )
    direct["comparison_type"] = "direct_DA_perfect_foresight"

    context_pairs = [
        ("SP_perfect_foresight", "WD rolling intrinsic"),
        ("WD60_perfect_foresight", "WD rolling intrinsic"),
        ("DA", "Forward simulation (LSMC)"),
    ]
    context_rows = []
    for stream, method in context_pairs:
        merged = b1[b1["stream"].eq(stream)].merge(
            nb13[nb13["nb13_method"].eq(method)],
            on="duration_h",
            how="inner",
        )
        merged["comparison_type"] = "context_not_strictly_like_for_like"
        context_rows.append(merged)

    comparison = pd.concat([direct, *context_rows], ignore_index=True)
    comparison["diff_gbp_annualized_m"] = (
        comparison["b1_value_gbp_annualized_m"] - comparison["nb13_value_gbp_annualized_m"]
    )
    comparison["diff_pct_vs_nb13"] = (
        comparison["diff_gbp_annualized_m"] / comparison["nb13_value_gbp_annualized_m"] * 100.0
    )

    return comparison[[
        "comparison_type",
        "duration_h",
        "stream",
        "nb13_method",
        "b1_gbp_per_mw_year_k",
        "b1_value_gbp_annualized_m",
        "nb13_gbp_per_mw_year_k",
        "nb13_value_gbp_annualized_m",
        "diff_gbp_annualized_m",
        "diff_pct_vs_nb13",
    ]]


comparison = build_nb13_comparison(df_rows)
out_cmp_csv = PROCESSED / "historical_lsmc_b1_vs_nb13.csv"
out_cmp_png = PROCESSED / "historical_lsmc_b1_vs_nb13.png"
comparison.to_csv(out_cmp_csv, index=False)
comparison.round(2)

In [ ]:
direct = comparison[comparison["comparison_type"].eq("direct_DA_perfect_foresight")]
plot = direct.set_index("duration_h")[[
    "b1_value_gbp_annualized_m",
    "nb13_value_gbp_annualized_m",
]].rename(columns={
    "b1_value_gbp_annualized_m": "nr20 actual DA PF",
    "nb13_value_gbp_annualized_m": "nr13 simulated DA PF",
})

ax = plot.plot(kind="bar", figsize=(7.5, 4.2))
ax.set_title("nr20 vs nr13: DA perfect-foresight benchmark")
ax.set_xlabel("Battery duration (hours)")
ax.set_ylabel("GBPm/year for 100 MW asset")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="", fontsize=8)
plt.tight_layout()
plt.savefig(out_cmp_png, dpi=140, bbox_inches="tight")
plt.show()

print(f"Saved: {out_cmp_csv}")
print(f"Saved: {out_cmp_png}")

## Stack All nr20 and nr13 Valuations

This section lists all three nr20 historical B1 valuations and all six nr13 valuation methods for 1h and 2h only, in the same unit: `GBPm/year for the 100 MW asset`. It also labels the closest like-for-like match and the main methodological difference.

Rows are sorted by battery duration first: all 1h valuations together, then all 2h valuations together. The nr13 base files contain five methods; the sixth chart method, `MODO style forward look`, is loaded from `phase4_modo_wd_rows.json`.

In [ ]:
def load_nr13_all_methods(durations_h: list[float] | None = None) -> pd.DataFrame:
    if durations_h is None:
        durations_h = [1.0, 2.0]

    rows = []
    for duration_h in durations_h:
        label = f"{duration_h:g}h"
        path = PROCESSED / f"phase4_method_comparison_{label}.csv"
        if not path.exists():
            print(f"Missing nr13 reference: {path}")
            continue
        frame = pd.read_csv(path)
        frame["duration_h"] = duration_h
        rows.append(frame)

    modo_path = PROCESSED / "phase4_modo_wd_rows.json"
    if modo_path.exists():
        modo = pd.read_json(modo_path)
        modo = modo[modo["duration_h"].isin(durations_h)].copy()
        rows.append(modo)
    else:
        print(f"Missing nr13 MODO reference: {modo_path}")

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


NR20_META = {
    "DA": {
        "market_basis": "Actual DA prices",
        "foresight": "Full historical perfect foresight",
        "policy_type": "Full-horizon deterministic LP",
        "scope": "Energy-only price path, VOM/degradation as configured",
        "strict_like_for_like_to": "nr13 Perfect foresight (DA energy)",
        "key_difference": "Actual historical DA path rather than HPFC-anchored simulated DA paths.",
    },
    "SP_perfect_foresight": {
        "market_basis": "Actual system prices",
        "foresight": "Full historical perfect foresight",
        "policy_type": "Full-horizon deterministic LP",
        "scope": "Energy-only price path, VOM/degradation as configured",
        "strict_like_for_like_to": "None; upper-bound context for imbalance value",
        "key_difference": "Uses realized system prices with perfect foresight; not a tradable nr13 policy.",
    },
    "WD60_perfect_foresight": {
        "market_basis": "Actual DA plus SP-DA spread capped at GBP60/MWh",
        "foresight": "Full historical perfect foresight",
        "policy_type": "Full-horizon deterministic LP",
        "scope": "Energy-only price path, VOM/degradation as configured",
        "strict_like_for_like_to": "None; context for WD/MODO volatility capture",
        "key_difference": "Uses realized imbalance spread capped at GBP60/MWh with perfect foresight.",
    },
}

NR13_META = {
    "Initial hourly intrinsic": {
        "market_basis": "HPFC hourly curve",
        "foresight": "Deterministic daily HPFC look-ahead",
        "policy_type": "Daily intrinsic LP",
        "scope": "Energy-only",
        "strict_like_for_like_to": "None; methodological context",
        "key_difference": "Deterministic forward curve, not historical actual prices.",
    },
    "DA rolling intrinsic": {
        "market_basis": "HPFC-anchored simulated DA paths",
        "foresight": "Rolling finite-window look-ahead",
        "policy_type": "Rolling intrinsic LP",
        "scope": "DA energy-only",
        "strict_like_for_like_to": "None; methodological context",
        "key_difference": "Rolling policy on simulated paths, not full historical perfect foresight.",
    },
    "WD rolling intrinsic": {
        "market_basis": "HPFC-anchored simulated WD paths",
        "foresight": "Rolling finite-window look-ahead",
        "policy_type": "Rolling intrinsic LP",
        "scope": "WD energy-only",
        "strict_like_for_like_to": "None; methodological context",
        "key_difference": "Rolling policy on simulated WD paths, not realized SP/WD perfect foresight.",
    },
    "MODO style forward look": {
        "market_basis": "HPFC-anchored simulated WD paths, GBP60/MWh WD cap",
        "foresight": "Rolling finite-window look-ahead",
        "policy_type": "Rolling intrinsic LP",
        "scope": "WD energy-only with MODO-style cap",
        "strict_like_for_like_to": "None; context for WD/MODO volatility capture",
        "key_difference": "Uses MODO-style WD cap on simulated paths; closest context to nr20 WD60 but not like-for-like.",
    },
    "Forward simulation (LSMC)": {
        "market_basis": "HPFC-anchored simulated full-stack paths",
        "foresight": "Non-anticipative learned policy",
        "policy_type": "Forward policy simulation",
        "scope": "Full stack: DA plus ancillary/BM modes",
        "strict_like_for_like_to": "None; methodological context",
        "key_difference": "Non-anticipative full-stack policy, not energy-only perfect foresight.",
    },
    "Perfect foresight (DA energy)": {
        "market_basis": "HPFC-anchored simulated DA paths",
        "foresight": "Full simulated-path perfect foresight",
        "policy_type": "Full-horizon deterministic LP",
        "scope": "DA energy-only upper bound",
        "strict_like_for_like_to": "nr20 DA",
        "key_difference": "Same LP style as nr20 DA, but on HPFC-anchored simulated DA paths.",
    },
}


def attach_metadata(frame: pd.DataFrame, meta: dict[str, dict[str, str]], key_col: str) -> pd.DataFrame:
    out = frame.copy()
    for col in ["market_basis", "foresight", "policy_type", "scope", "strict_like_for_like_to", "key_difference"]:
        out[col] = out[key_col].map(lambda x: meta.get(x, {}).get(col, ""))
    return out

In [ ]:
COMPARISON_DURATIONS_H = [1.0, 2.0]

nr20_stack = df_rows[df_rows["duration_h"].isin(COMPARISON_DURATIONS_H)].copy()
nr20_stack["source"] = "nr20"
nr20_stack["valuation"] = nr20_stack["stream"]
nr20_stack["unit"] = "GBPm/year for 100 MW"
nr20_stack["value_gbp_annualized_m"] = nr20_stack["gbp_per_mw_year_k"] * POWER_MW / 1000.0
nr20_stack = attach_metadata(nr20_stack, NR20_META, "valuation")

nr13_stack = load_nr13_all_methods(COMPARISON_DURATIONS_H).copy()
nr13_stack["source"] = "nr13"
nr13_stack["valuation"] = nr13_stack["method"]
nr13_stack["unit"] = "GBPm/year for 100 MW"
nr13_stack = attach_metadata(nr13_stack, NR13_META, "valuation")

stack_cols = [
    "source",
    "valuation",
    "duration_h",
    "unit",
    "value_gbp_annualized_m",
    "gbp_per_mw_year_k",
    "market_basis",
    "foresight",
    "policy_type",
    "scope",
    "strict_like_for_like_to",
    "key_difference",
]
stacked = pd.concat([nr20_stack[stack_cols], nr13_stack[stack_cols]], ignore_index=True)

valuation_order = {
    "DA": 0,
    "SP_perfect_foresight": 1,
    "WD60_perfect_foresight": 2,
    "Initial hourly intrinsic": 3,
    "DA rolling intrinsic": 4,
    "WD rolling intrinsic": 5,
    "MODO style forward look": 6,
    "Forward simulation (LSMC)": 7,
    "Perfect foresight (DA energy)": 8,
}
stacked["_valuation_order"] = stacked["valuation"].map(valuation_order).fillna(99)
stacked = stacked.sort_values(["duration_h", "_valuation_order", "source"])
stacked = stacked.drop(columns=["_valuation_order"]).reset_index(drop=True)

out_stack_csv = PROCESSED / "historical_lsmc_b1_nr13_stacked_table.csv"
stacked.to_csv(out_stack_csv, index=False)

stacked[[
    "source",
    "valuation",
    "duration_h",
    "value_gbp_annualized_m",
    "strict_like_for_like_to",
    "key_difference",
]].round(2)

In [ ]:
stacked_wide = (
    stacked.pivot_table(
        index=[
            "source",
            "valuation",
            "unit",
            "market_basis",
            "foresight",
            "policy_type",
            "scope",
            "strict_like_for_like_to",
            "key_difference",
        ],
        columns="duration_h",
        values="value_gbp_annualized_m",
        aggfunc="first",
    )
    .rename(columns=lambda c: f"value_{c:g}h_gbp_m_per_year")
    .reset_index()
)

out_stack_wide_csv = PROCESSED / "historical_lsmc_b1_nr13_stacked_wide.csv"
stacked_wide.to_csv(out_stack_wide_csv, index=False)

stacked_wide.round(2)

In [ ]:
# Visual stacked list: one horizontal bar per valuation/duration row.
plot_df = stacked.copy()
plot_df["label"] = plot_df["source"] + " | " + plot_df["valuation"] + " | " + plot_df["duration_h"].map(lambda x: f"{x:g}h")
plot_df["_valuation_order"] = plot_df["valuation"].map(valuation_order).fillna(99)
plot_df = plot_df.sort_values(["duration_h", "_valuation_order", "source"]).drop(columns=["_valuation_order"])

fig_h = max(5.5, 0.28 * len(plot_df))
ax = plot_df.plot.barh(
    x="label",
    y="value_gbp_annualized_m",
    figsize=(10, fig_h),
    legend=False,
    color=np.where(plot_df["source"].eq("nr20"), "#1f77b4", "#7f7f7f"),
)
ax.set_title("Stacked nr20 and nr13 valuations: 1h then 2h")
ax.set_xlabel("GBPm/year for 100 MW asset")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

out_stack_png = PROCESSED / "historical_lsmc_b1_nr13_stacked.png"
plt.savefig(out_stack_png, dpi=140, bbox_inches="tight")
plt.show()

print(f"Saved: {out_stack_csv}")
print(f"Saved: {out_stack_wide_csv}")
print(f"Saved: {out_stack_png}")